# Milestone 4 Lab, notebook 4 of 7: Two runs compared

## Where we are

| Notebook | What you do |
|---|---|
| 1. A figure, traced by hand | Find every record behind one figure yourself, and add them up |
| 2. Run records | Give every pipeline run an id and a record, and keep every run |
| 3. Lineage | Save which records made each figure, and prove it adds up |
| **4. Two runs compared** (you are here) | Find what changed between two runs: settings, inputs, figures, records |
| 5. A trace of the analyst | Give the analyst the new tools, and record every step it takes |
| 6. Scoring an agent | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| 7. Two experiments | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **0**. A full run of this lab costs about 5 US cents.

## What this notebook is for

You now have runs with records, and lineage in every run. This notebook uses them to answer the second question from notebook 1: **why did this number change?**

In notebook 1 you answered it by guessing and testing the guess. Here you look it up. You compare two runs piece by piece, by hand, and then with the production function that does the same.

No AI is used in this notebook. A difference between two figures is a calculation, and in this project a model never calculates.

> Run notebook 2 first. This notebook reads the runs it made.

In [ ]:
import shutil

import pandas as pd
from lab_helpers import (
    LAB_SOURCE,
    LAB_WAREHOUSE,
    SOURCE,
    changed_figures_by_hand,
    load_result,
    read_run_table,
    show_doc,
)

from retail_profit_stack.lineage.runs import read_run_record

## Step 1: The two runs

In [ ]:
show_doc("HLD.md", "6. Two runs to compare")

In [ ]:
earlier = read_run_record("RUN-0001", LAB_WAREHOUSE)
later = read_run_record("RUN-0002", LAB_WAREHOUSE)

print("Earlier:", earlier["run_id"], "-", earlier["note"])
print("Later:  ", later["run_id"], "-", later["note"])

## Step 2: Did a setting change?

A figure can only move if something that feeds it moved. There are three candidates: the settings, the inputs, and the code. Check each in turn, starting with the settings.

In [ ]:
for name in earlier["settings"]:
    before = earlier["settings"][name]
    after = later["settings"][name]
    marker = "<-- changed" if before != after else ""
    print(f"{name:12} {before!s:8} {after!s:8} {marker}")

## Step 3: Did an input change?

Compare the fingerprint of each source table.

In [ ]:
for table in earlier["inputs"]:
    before = earlier["inputs"][table]["fingerprint"]
    after = later["inputs"][table]["fingerprint"]
    marker = "<-- changed" if before != after else ""
    print(f"{table:15} {before}  {after}  {marker}")

## Step 4: Did the code change?

The same again, for each transformation and each check.

In [ ]:
changed_code = []
for name in earlier["code"]:
    if earlier["code"][name] != later["code"][name]:
        changed_code.append(name)

print(len(earlier["code"]), "code files compared. Changed:", changed_code)

So the inputs are the same and the code is the same. **One setting changed.** That is already the reason, read straight from the records.

## Step 5: Which figures moved?

Put the two profit tables side by side and keep the rows where contribution profit differs.

In [ ]:
earlier_profit = read_run_table("RUN-0001", "monthly_product_profit")
later_profit = read_run_table("RUN-0002", "monthly_product_profit")

both = earlier_profit.merge(later_profit, on=["month", "product_id"], suffixes=("_earlier", "_later"))

both["change"] = both["contribution_profit_later"] - both["contribution_profit_earlier"]
both["change"] = both["change"].round(2)

moved = both[both["change"].abs() > 0.005]
moved[["month", "product_id", "contribution_profit_earlier", "contribution_profit_later", "change"]]

Three figures moved out of 24. The second row is the one from notebook 1: $5,188.20 to $5,156.94.

Check it against what you worked out by hand.

In [ ]:
lineage_key = load_result("answer_key_lineage", required_from="notebook 1")

bottle = moved[(moved["product_id"] == "P001") & (moved["month"] == "2026-10")].iloc[0]

print("Change from the two runs:", bottle["change"])
print("Change by hand:          ", lineage_key["change"])

Look at the first row too. The two-person tent's profit went **up** by $334.00. Would you have guessed that setting duplicates aside could raise a profit? The next step shows why.

## Step 6: Which records came or went?

Each run has its own lineage. A record that is in the earlier run's lineage and not in the later one's is a record that stopped counting.

In [ ]:
columns = ["month", "product_id", "source_table", "row_number"]

earlier_records = read_run_table("RUN-0001", "figure_lineage")[columns].drop_duplicates()
later_records = read_run_table("RUN-0002", "figure_lineage")[columns].drop_duplicates()

print("Records counted in RUN-0001:", len(earlier_records))
print("Records counted in RUN-0002:", len(later_records))

`merge` with `indicator=True` adds a column that says where each record was found: in the left table only, the right table only, or both.

In [ ]:
side_by_side = earlier_records.merge(later_records, on=columns, how="outer", indicator="found_in")

only_earlier = side_by_side[side_by_side["found_in"] == "left_only"]

print("Records only in the earlier run:", len(only_earlier))
only_earlier.groupby(["month", "product_id", "source_table"]).size()

22 records stopped counting, for exactly the three products whose figures moved.

And there is the tent: one **refund** stopped counting. A refund takes money off. It had been taken off twice. Now it is taken off once, so the profit is higher.

## Step 7: Which check set them aside?

The later run's findings name the check.

In [ ]:
findings = read_run_table("RUN-0002", "quality_findings")

findings[findings["outcome"] == "quarantine"][["finding_id", "check_name", "month", "product_id", "record_count"]]

The record counts match step 6: 1, 9, and 12 make 22.

You now have the full answer, with nothing guessed:

> The trail water bottle's October contribution profit fell by $31.26 between `RUN-0001` and `RUN-0002`. The inputs and the code were the same. One setting changed: duplicated records are now set aside. Nine of its records (three order lines, three shipments, three fees) were set aside by the `duplicate_records` check, finding QF-002.

## Step 8: The production function

`compare_runs` does steps 2 to 7 in one call and returns each part as a list.

In [ ]:
from retail_profit_stack.lineage.compare import compare_runs

comparison = compare_runs("RUN-0001", "RUN-0002", LAB_WAREHOUSE)

for part, value in comparison.items():
    size = len(value) if isinstance(value, list) else value
    print(f"{part:18} {size}")

In [ ]:
print("Settings changed:", comparison["settings_changed"])
print("Inputs changed:  ", comparison["inputs_changed"])
print("Code changed:    ", comparison["code_changed"])
print("Checks changed:  ", comparison["checks_changed"])

`figures_changed` has every term that moved, not only contribution profit. Filter it to compare with your table from step 5.

In [ ]:
figures = pd.DataFrame(comparison["figures_changed"])
figures[figures["metric"] == "contribution_profit"]

In [ ]:
by_hand = changed_figures_by_hand(earlier_profit, later_profit)
by_hand

`changed_figures_by_hand` is your step 5, kept in `lab_helpers.py`. A test compares it with production.

## Step 9: Two runs with nothing changed

In notebook 2 you made `RUN-0003` with nothing changed. The comparison should be empty.

In [ ]:
nothing = compare_runs("RUN-0002", "RUN-0003", LAB_WAREHOUSE)

for part in ["settings_changed", "inputs_changed", "code_changed", "figures_changed", "records_changed"]:
    print(f"{part:18} {nothing[part]}")

## Step 10: The other kind of change

So far a **setting** changed. Now change an **input**.

Remember the camp stove: it has no cost for October, so its profit is unavailable. Suppose the vendor finally sends the cost. We add it to a **copy** of the records and run the pipeline on the copy.

In [ ]:
if LAB_SOURCE.exists():
    shutil.rmtree(LAB_SOURCE)
shutil.copytree(SOURCE, LAB_SOURCE)

with (LAB_SOURCE / "product_costs.csv").open("a", encoding="utf-8") as file:
    file.write("P007,2026-10,44.00\n")

print((LAB_SOURCE / "product_costs.csv").read_text(encoding="utf-8")[-60:])

In [ ]:
from retail_profit_stack.transform.pipeline import run_pipeline

fourth_run = run_pipeline(
    "sql",
    source_dir=LAB_SOURCE,
    warehouse_dir=LAB_WAREHOUSE,
    note="The camp stove's October cost has arrived",
)

print(fourth_run.run_id)

In [ ]:
after_cost = compare_runs("RUN-0003", "RUN-0004", LAB_WAREHOUSE)

print("Settings changed:", after_cost["settings_changed"])
print("Inputs changed:  ", after_cost["inputs_changed"])
print("Checks changed:  ", after_cost["checks_changed"])

This time no setting changed. One input's fingerprint did: `product_costs`. And the `missing_product_cost` check, which needed review before, now passes.

In [ ]:
pd.DataFrame(after_cost["figures_changed"])

In [ ]:
pd.DataFrame(after_cost["records_changed"])

Three figures became available, and one record is new in the later run: the cost row we added. `change` is empty, because "unavailable" has no amount to subtract from.

This run lives only in the lab folder. The real history still has two runs, and the camp stove's cost is still missing there.

## What we now know

- A figure moves only when a **setting**, an **input**, or the **code** moves. The run records show which.
- Comparing two runs is subtraction on the figures and set difference on the lineage. Code does both.
- Between `RUN-0001` and `RUN-0002`: one setting changed, three figures moved, and 22 records stopped counting. One profit went **up**, because a duplicated refund is no longer taken off twice.
- We tried both kinds of change: a setting (duplicates set aside) and an input (a cost arrived).
- Nothing here was guessed. Compare that with notebook 1, step 7.

**Next:** notebook 5. The analyst gets a tool that returns this comparison, and we start recording what the analyst itself does.